# Pipeline 07: SLM Distillation & Edge Export

**Goal:** Fine-tune Qwen 2.5 1.5B to replicate Claude's parsing capabilities. Export to GGUF for offline/mobile inference.

**Depends on:** Pipeline 06 (training dataset)

**Runtime:** GPU required (T4 minimum for QLoRA). A100 recommended for speed.

In [2]:
# Setup
!pip install unsloth -q
!pip install --no-deps xformers trl peft accelerate bitsandbytes -q

from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil, torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

PROJECT = '/content/drive/MyDrive/photo-style-rl'
DATASET_PATH = f'{PROJECT}/data/slm_training_dataset.jsonl'
OUTPUT_DIR = f'{PROJECT}/models/qwen_fuji_v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Schema constants + validate_payload imported from shared.py — same gate as NB06.
# Defense-in-depth: the dataset was already validated at generation time, but we
# re-validate here to catch any manually added or corrupted records before training.
shutil.copy(f'{PROJECT}/src/shared.py', '/content/shared.py')
from shared import (
    VALID_TONE_CURVE_KEYS, VALID_COLOR_NAMES, VALID_HSL_KEYS, VALID_BASIC_KEYS,
    validate_payload,
)

print("Ready.")


Mounted at /content/drive
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Ready.


In [3]:
# Data sanitization (strict) — defense-in-depth validation before training.
# validate_payload() and schema constants are imported from shared.py (see Cell 1).
# This catches any records that slipped through NB06 or were added manually.

valid_records = []
total = 0
with open(DATASET_PATH, 'r') as f:
    for line in f:
        total += 1
        record = json.loads(line)
        try:
            payload = json.loads(record['messages'][1]['content'])
            if validate_payload(payload):
                valid_records.append(record)
        except:
            pass

print(f"Loaded {len(valid_records)} valid records from {total} total ({total - len(valid_records)} rejected)")

if len(valid_records) < 50:
    print("⚠️ Fewer than 50 clean records. Run Pipeline 06 to generate more data.")

dataset = Dataset.from_list(valid_records)


Loaded 314 valid records from 334 total (20 rejected)


In [4]:
# Model loading + tokenization
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def formatting_func(examples):
    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)
print(f"Model loaded. Dataset tokenized: {len(dataset)} records.")

==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Map:   0%|          | 0/314 [00:00<?, ? examples/s]

Model loaded. Dataset tokenized: 314 records.


In [5]:
# QLoRA training
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# Scale max_steps to dataset size
# Rule of thumb: 2-3 epochs worth of steps
steps_per_epoch = max(len(dataset) // (2 * 4), 1)  # batch_size=2, grad_accum=4
max_steps = steps_per_epoch * 3
print(f"Training for {max_steps} steps (~3 epochs over {len(dataset)} records)")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=max_steps,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Starting fine-tuning...")
stats = trainer.train()
print(f"Done. Peak GPU memory: {round(torch.cuda.max_memory_reserved() / 1e9, 2)} GB")

Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Training for 117 steps (~3 epochs over 314 records)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/314 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 314 | Num Epochs = 3 | Total steps = 117
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,3.215151
2,2.894014
3,3.238668
4,2.947826
5,2.568144
6,2.257420
7,2.082833
8,1.872850
9,1.648338
10,1.591415


Done. Peak GPU memory: 3.09 GB


In [6]:
# Export to GGUF
export_path = f"{OUTPUT_DIR}/qwen_fuji_q4_k_m"

model.save_pretrained_gguf(
    export_path,
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"\nExported to: {export_path}-unsloth.Q4_K_M.gguf")
print("Drop this into LM Studio, Ollama, or MLX for offline inference.")
print("\nFull pipeline complete:")
print("  01: Text → Params (Claude API)")
print("  02: Style Profiling + SAM Region Detection")
print("  03: Advanced Color Science Extraction")
print("  04: End-to-End Inference Demo")
print("  05: Interactive UI + Data Flywheel")
print("  06: Synthetic Data Distillation")
print("  07: Qwen Fine-Tuning + Edge Export")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 939.79it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:25<00:00, 145.12s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/qwen2.5-1.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/qwen2.5-1.5b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/qwen2.5-1.5b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/drive/MyDrive/photo-